In [1]:
import sys
sys.path.append('../')

%env MUJOCO_GL=egl

env: MUJOCO_GL=egl


In [2]:
import jax
import jax.numpy as jnp
import mujoco
import numpy as np
import mediapy as media
from dataclasses import dataclass, field
from mujoco.mjx._src import math as mjx_math

from builderbench.env_utils import make_env
from utils.wrapper import wrap_env

In [3]:
@dataclass
class Args:
    # experiment
    agent: str = "mp"
    seed: int = 1

    # environment
    env_id: str = 'sparse-creative-4-task1'
    env_early_termination: bool = True
    env_episode_length: int = None
    permutation_invariant_reward: bool = True   # invariance to the order of cubes in any structure

    # planner
    kp_pos: float = 10.0
    kd_pos: float = 2.0

    kp_yaw: float = 10.0
    kd_yaw: float = 0.5
    
    num_envs: str = 1

In [4]:
args = Args()

In [5]:
env_class, default_config = make_env(args)
env = env_class(config=default_config)

Warp 1.9.0 initialized:
   CUDA Toolkit 12.8, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
     "cuda:1"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
     "cuda:2"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
     "cuda:3"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
     "cuda:4"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
     "cuda:5"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
     "cuda:6"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
     "cuda:7"   : "NVIDIA A100-SXM4-80GB" (79 GiB, sm_80, mempool enabled)
   CUDA peer access:
     Supported fully (all-directional)
   Kernel cache:
     /home/nvidia/.cache/warp/1.9.0


In [6]:
np.random.seed(args.seed)
key = jax.random.PRNGKey(args.seed)
key, key_env, key_eval, key_policy, key_value = jax.random.split(key, 5)

In [7]:
reset_fn = jax.jit(env.reset)
step_fn = jax.jit(env.step)

In [8]:
@jax.jit
def get_yaw_from_quat(q):
    w, x, y, z = q[0], q[1], q[2], q[3]
    siny_cosp = 2 * (w * z + x * y)
    cosy_cosp = 1 - 2 * (y * y + z * z)
    yaw = jnp.arctan2(siny_cosp, cosy_cosp)
    return yaw

@jax.jit
def normalize_angle(angle):
    return jnp.arctan2(jnp.sin(angle), jnp.cos(angle))

In [9]:
@jax.jit
def get_waypoint(env_state, cube_id):
    current_pos = env_state.obs[:3*env._config.num_cubes].reshape(env._config.num_cubes, 3)[cube_id]
    current_quat = env_state.obs[3*env._config.num_cubes:][:4*env._config.num_cubes].reshape(env._config.num_cubes, 4)[cube_id]
    current_linvel = env_state.obs[7*env._config.num_cubes:][:3*env._config.num_cubes].reshape(env._config.num_cubes, 3)[cube_id]
    current_angvel = env_state.obs[10*env._config.num_cubes:][:3*env._config.num_cubes].reshape(env._config.num_cubes, 3)[cube_id]
    
    target_pos = env_state.info['target_goal'].reshape(env._config.num_cubes, 3)[cube_id]
    
    current_xy = current_pos[:2]
    target_xy = target_pos[:2]
    
    current_height = current_pos[-1]
    top_height = target_pos[-1] + 0.05
    
    horizontal_dist_to_target = jnp.linalg.norm(current_xy - target_xy)
    dist_to_target = jnp.linalg.norm(current_pos - target_pos)

    is_far = horizontal_dist_to_target > 0.01
    is_low = current_height < (top_height - 0.005)

    wp_lift = target_pos.at[:2].set(current_xy).at[2].set(top_height)
    wp_hover = target_pos.at[2].set(top_height)
    wp_final = target_pos
        
    current_waypoint = jnp.where(
        is_far,
        jnp.where(is_low, wp_lift, wp_hover),
        wp_final
    )

    return current_waypoint, {'dist_to_target':dist_to_target, 'target_pos': target_pos}

@jax.jit
def get_action(env_state, waypoint, cube_id):
    current_pos = env_state.obs[:3*env._config.num_cubes].reshape(env._config.num_cubes, 3)[cube_id]
    current_quat = env_state.obs[3*env._config.num_cubes:][:4*env._config.num_cubes].reshape(env._config.num_cubes, 4)[cube_id]
    current_linvel = env_state.obs[7*env._config.num_cubes:][:3*env._config.num_cubes].reshape(env._config.num_cubes, 3)[cube_id]
    current_angvel = env_state.obs[10*env._config.num_cubes:][:3*env._config.num_cubes].reshape(env._config.num_cubes, 3)[cube_id]
    
    current_yaw = get_yaw_from_quat(current_quat)
    
    error_pos = waypoint - current_pos
    output_pos = (args.kp_pos * error_pos) + (args.kd_pos * - current_linvel)
    output_pos = output_pos.at[2].add(gravity_comp)
    
    error_yaw = normalize_angle(0.0 - current_yaw)
    output_yaw = (args.kp_yaw * error_yaw) + (args.kd_yaw * - current_angvel[-1])

    raw_ctrl_action = jnp.concatenate([output_pos, output_yaw[None]], axis=0)
    ctrl_action = ( raw_ctrl_action - env._ctrl_median[:4] ) / env._ctrl_halfspan[:4]
    
    select_action = ( ( ( 2 * cube_id + 1) * jnp.pi / env._config.num_cubes ) - jnp.pi ) / ( jnp.pi )

    action =  jnp.concatenate([ctrl_action, select_action[None]], axis=0)
    action = jnp.clip(action, -1, 1)
    
    return action

In [10]:
cube_mass = 0.07936
gravity = 9.81
gravity_comp = cube_mass * gravity

In [11]:
camera = mujoco.MjvCamera()
camera.distance = 0.8
camera.lookat = np.array([0.4, 0.0 , 0.4])
camera.elevation = -30.0
camera.azimuth = 180

In [12]:
default_config.episode_length

500

In [13]:
cube_id  = 0
rollout = []
returns = []
env_state = reset_fn(key_env)
rollout.append(env_state)

for i in range(default_config.episode_length):
    wp, wp_info = get_waypoint(env_state, cube_id)
    action = get_action(env_state, wp, cube_id)

    if wp_info['dist_to_target'] < 0.01:
        cube_id = np.clip( cube_id + 1, a_min=0, a_max=env._config.num_cubes)

    env_state = step_fn(env_state, action)
    rollout.append(env_state)

    returns.append( env_state.reward )
        
    # print(f'cube id is {cube_id} and target position is {wp_info["target_pos"]}')
    # print(f'cube id is {cube_id} and action taken is {jnp.round(action, 6)}')

Module mujoco.mjx.third_party.mujoco_warp._src.smooth f8aea15 load on device 'cuda:0' took 7.31 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.collision_driver abf11d9 load on device 'cuda:0' took 0.46 ms  (cached)
Module _nxn_broadphase__locals__kernel_693f1c65 693f1c6 load on device 'cuda:0' took 0.41 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.collision_primitive._create_narrowphase_kernel 64aeafa load on device 'cuda:0' took 1.10 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.constraint 314db63 load on device 'cuda:0' took 1.88 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.forward ea0168c load on device 'cuda:0' took 0.91 ms  (cached)
Module _create_actuator_velocity_kernel__locals__actuator_velocity_4b9adcc9 9173f9a load on device 'cuda:0' took 0.44 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.passive c66cecd load on device 'cuda:0' took 0.85 ms  (cached)
Module mujoco.mjx.third_party.mujoco_warp._src.support 7

In [14]:
print(np.sum( returns ) )

-974


In [15]:
default_config.episode_length

500

In [16]:
video_images = []
mocap_key = 'target_mocap'
for i in range(default_config.episode_length):
    if i % 2 == 0:
        video_images.append(
            env.render_from_info(
                rollout[i].data.qpos,
                rollout[i].data.qvel, 
                rollout[i].info[f'{mocap_key}_pos'],
                rollout[i].info[f'{mocap_key}_quat'],
                camera=camera,
            )
        )

KeyboardInterrupt: 

In [118]:
media.show_video(video_images, fps=1.0 / env.dt / 2)

This browser does not support the video tag.
